# Introduction

ADN = séquence de base 

on compare toujours à une reference ex GRCh38 

Un variant = différence par rapport à la référence

Type de variant = 
- SNP (substition)
- insertion
- deletion
- indel (combinaison ins_del)

Un variant peut être :
* germinal (toutes cellules)
* somatique (clone tumoral)

In [ ]:
ref = "ATGCCGTA"
pat = "ATGACGTT"

mut = 0

for i in range(len(ref)):
    if ref[i] != pat[i]:
        mut += 1

print(mut)

Le NGS produit des fragments d'ADN appelé reads 

Lire la même base 100 fois = couverture 100X

Plus la couverture est élevée :

- plus fiable
- meilleure détection faible VAF
- moins de bruit

MRD (Minimal Residual Disease).

Ça signifie :

la majorité des cellules leucémiques ont disparu
mais il reste un petit clone détectable
risque de rechute

Exemple :

*diagnostic : 40%
*après traitement : 0.3%
*→ MRD 

Une mutation **germinale** est présente dans toutes les cellules de l’organisme.

Exemples :

* sang
* peau
* salive
* moelle
* → même mutation partout

Parce qu’elle est héritée (ou apparue très tôt).

Une mutation **somatique** est présente seulement dans un clone de cellules (ex. cellules leucémiques).

* Elle apparaît au cours de la vie
* Elle n’est pas héritée
* Elle peut disparaître après traitement

# BASH 

* **awk** = outil qui lit un fichier ligne par ligne 

*  Notion $0 $1 $2 
 
- $0 = toute la ligne
- $1 = premier mot
- $2 = deuxième mot

* Notion NR 

NR = numéro de ligne 

* Socker une variable 

s=$0


In [ ]:
# affiche tout les ligne d'un txt
awk '{print $0}' test.txt
# affiche le premier mot de chaque txt
awk '{print $1}' test.txt
# affiche sur un cycle de 4 la 2e ligne 
awk 'NR%4==2' test
# stocker variable
NR%4==1 {h=$0}
NR%4==2 {s=$0}
NR%4==3 {p=$0}
NR%4==0 {q=$0}
# la sequence s contient ATG 
s ~ /ATG/
# la sequence ne contient pas ATG 
s !~ /ATG/
# la sequence s contient ATG ou TTT 
s ~ /ATG|TTT/
s ~ /ATG/ || s ~ /TTT/
# la sequence s contient ATG et TTT
s ~ /ATG/ && s ~ /TTT/

# Affichage 
print h "\n" s "\n" p "\n" q
    #\n retour a la ligne 


# si la sequence contient /ATG/ garde le 4 ligne du read 
awk '
NR%4==1 {h=$0}
NR%4==2 {s=$0}
NR%4==3 {p=$0}
NR%4==0 {q=$0
    if ( s ~ /ATG/ && s !~ /N/) 
        print h "\n" s "\n" p "\n" q
}
' reads.fastq


# FastQ


## Introduction

Toutes les analyses NGS commencent par ce fichier.

Quand une machine NGS séquence un ADN :

Elle lit des millions de fragments.

Chaque fragment = une lecture (read).

Chaque lecture contient :

*  la séquence
*  la qualité de chaque base

Tout ça est stocké dans un fichier :

FASTQ


In [ ]:
Stucture d'une lecture FASQ :

@SEQ_ID    /Identifiant 
ATCGTTAGCTAG   /Séquence d'ADN 
+                    /séparateur
IIIIIIIIIIII               /Qualité 

Si on parcout un FAST Q ligne par ligne, pour identifier les lignes de séquence, la condition sera : 

```
index % 4 == 1
```

* **Score Phred** (code ASCII)

| caractère | qualité       |
| --------- | ------------- |
| `!`       | très mauvaise |
| `H`       | bonne         |
| `I`       | très bonne    |

pour transformer en python lettre en nombre ```ord("I")```

👉 qualité réelle = ord(caractère) - 33

score **Phred** = 40 = très bonne qualité




## L'Oracle FASTQC 

Outil universel de QC 

QC = évaluer la qualité des données avant analyse

Il analyse les FASTQ et génère des graphiques de qualité.

FastQC vérifie plusieurs chose : 
1. La qualité des bases
2. GC content : vérifie la proportion G+C est normal
3. Distribution des longueurs : les reads doivent avoir les mêmes tailles 
4. Adapteur : parfois la machine lit ADN + adaptateur, les adaptateurs doivent être retirés 


## Trim reads

Trimmer des reads signifie : couper certaines parties des lectures FASTQ avant l’alignement

**Trimming** = couper les parties de mauvaise qualité 

On enlève généralement :

- bases de mauvaise qualité
- adaptateurs de séquençage
- parfois les extrémités trop bruitées


Les outils bioinformatiques utilisés :
- Trimmomatic
- Cutadapt
- fastp  (très populaire)
- Trim Galore



## Flitering 

Flitering = supprimer les reads de mauvaise qualité

## code pytho et shell 

In [ ]:
with open("reads.fastq", "r") as f:
    while True:
        h = f.readline().strip()
        if not h:  # fin du fichier
            break
        s = f.readline().strip()
        p = f.readline().strip()
        q = f.readline().strip()

        if not "I" in s:
            continue
        else : 
            print(h)
            print(s)
            print(p)
            print(q)

In [ ]:
awk '
NR%4==1 {h=$0}
NR%4==2 {s=$0}
NR%4==3 {p=$0}
NR%4==0 {q=$0
    if ( s ~ /ATG/ && s !~ /N/) 
        print h "\n" s "\n" p "\n" q
}
' reads.fastq

# Alignement (mapping) des lectures / BAM

Quand on séquence on obtient donc des reads mais on ne sait pas où ces fragments se trouvent dans le génome humain.

L'alignement sert à placer chaque read sur le génome de référence.

En NGS les outils d'alignements les plus utilisés sont : 
- BWA (très utilisé)
- Bowtie2 
- STAR (RNA-seq)

Le résultat de cet alignement produit un fichier qui s'appelle un BAM.

Il y a différent problème qu'on peut avoir lors de l'alignement :
- le génome est très grand
- erreur de séquençage
- régions répétées dans le génome
- présence d’un variant réel
- mauvaise qualité des bases

Lors de problèmes de **multi-mapping reads**, l'aligner peut : 
- choisir une position probable 
- marquer la read comme multi-alignée
- ou la rejeter

## BAM 

Un fichier BAM est un fichier binaire contenant les reads alignées. 
C'est la version compréssée du fichier SAM. 

On utilise en bioinformatique les BAM car les données NGS sont énormes, un FASTQ peut faire 20 à 80 Go. Cette version binaire nous permet donc de réduire la taille des fichiers, lecture plus rapide par les logiciels et permettre l'indexation du fichier 

Un BAM contient : 
- séquence
- position sur le génome 
- score d'alignement 
- orientation 
- qualité 

Les outils utilisé qui manipulent les BAM sont : 
- samtools 
- IGV (visualisation) 
- GATK 

# VCF

## Notation VCF

**DP=Depth** => nombre total de reads à cette position 

**AD=Allelic Depth** => nombre de reads pour chaque allèle *AD = REF,ALT*

## VAF 

La VAF signifie : Varaint Allele Frequency 

C'est la proportion de reads qui portent un variant à une postion donnée 

```
VAF = nombre de reads avec le variant
      --------------------------------
      nombre total de reads à cette position
```

La VAF donne une information sur la proportion de cellules qui portent la mutation.

**Mutation hétérozygote dans toutes les cellules**

Chaque cellule possède :
- 1 allèle muté
- 1 allèle normal

Donc : VAF ≈ 50 %

**Mutation présente seulement dans une partie des cellules (clone tumoral)**

Exemple : 20 % des cellules mutées

Alors la VAF sera plus faible : VAF ≈ 10–20 %  (selon la zygosité)

Dans les hémopathies on a souvent :

- des clones cellulaires
- plusieurs sous-clones

La VAF aide à comprendre :

- la taille du clone muté
- l’évolution clonale
- la réponse au traitement

| VAF   | interprétation    |
| ----- | ----------------- |
| <1%   | bruit possible    |
| 1–5%  | zone grise        |
| ~10%+ | mutation probable |
| ~50%  | clone principal   |


* **EXEMPLE**

DP = 100 /
AD = 80,20 /
REF = A /
ALT = T

*Interprétation* : 
- 80 reads → A
- 20 reads → T
- VAF = 20%